# General feedback on assignments - good practices in python

### 1) Error / exception handling

Note the nuance: Error = issue with the syntax or program logic that will stop the execution. Exception = Raised internally to change the flow of a program execution instead of stopping it completely.

- Catch specific exceptions (ValueError, FileNotFoundError), not a bare except (hides real bugs, including typos and Ctrl+C)
- Keep the try block small: only wrap the line(s) that can actually fail.
- Use with for files so they close automatically, even on errors.
- Raise your own errors when input is invalid (see example below).
- Catch several types at once if you handle them the same way: except (ValueError, TypeError):
- Get the error message with as: except ValueError as e: print(e)

In [4]:
def read_number(text):
    try:
        return int(text)
    except ValueError as e:
        # print(f"'{text}' is not a valid number")
        print(e)
        return None


In [2]:
read_number("42")    # returns 42

42

In [5]:
read_number("abc")   # prints message, returns None

invalid literal for int() with base 10: 'abc'


Find out more about built-in exceptions in Python 3: https://docs.python.org/3/builtins/exceptions.html 

### 2) Naming conventions

There are naming conventions out there, e.g. in the official style guide PEP8:

- Functions are verbs, variables are nouns: load_data(), save_report(), is_valid() vs. sample_count, results
- Booleans read like questions: is_empty, has_header, can_retry
- Be descriptive, not cryptic: retention_time beats rt, and temp beats t. Short names like i, x are fine for tiny loops or math.
- Don’t shadow built-ins: avoid list, dict, str, sum, max, id, type as variable names. Use values, data_dict, total instead.
- Avoid confusing single letters: l, O, I look like 1 and 0.
- Be consistent within a project; if a codebase does something differently, follow the codebase.

Of course you can abbreviate, but make sure that someone else can still understand and read the code without your insider's knowledge.

### 3) False and None
These are not interchangeable!!

False is a boolean value, meaning "no".

None is a data type and means "no value".

In [6]:
print(type(False))
print(type(None))

<class 'bool'>
<class 'NoneType'>


In [7]:
False == None

False

You check for `None` with `is`, not `==`!

In [8]:
if result is None:
    print("no result")

NameError: name 'result' is not defined

Typical use of `None`: a function that has nothing to return, or a default for "not provided", e.g.

In [ ]:
def find_user(name, users):
    for user in users:
        if user == name:
            return user
    return None    # not found (also happens implicitly if you omit it)

Good practices:

- Use ``is None`` / ``is not None`` when you mean “no value”.
- Use ``False``/``True`` only for real yes/no flags.
- Don’t write ``== True`` or ``== False`` in conditional logic; just use ``if flag:`` / ``if not flag:``.
- Never use ``None`` as a mutable default (``def f(items=[])``); use ``items=None`` and create the list inside.

### 4) Pass vs. return

``pass`` does nothing (it’s a placeholder), not even stop the code - the compiler will simply go to the next line. ``return`` exits a function and optionally hands back a value, code after it never runs!

In [ ]:
def name():
    pass

In [ ]:
def f():
    print("some statement")
    pass
    print("still runs")   # this executes

f()

some statement
still runs


In [10]:
def f():
    print("some statement")
    return
    print("still runs")   # this is ignored

f()

some statement


One potential pitfall: `print` instead of ``return``:

In [11]:
def add_print(a, b):
    print(a + b)      # shows it, but returns None


x = add_print(2, 3)   # prints 5, but x is None

5


In [13]:
type(x)

NoneType

In [14]:
def add_return(a, b):
    return a + b      # gives the value back so you can use it

y = add_return(2, 3)  # y is 5, can be used further

In [16]:
type(y)

int

### 5) Absolute and relative paths

Relative paths are always more transferable!! In Jupyter Notebooks you can check for the present working directory using `pwd` (as Jupyter can make use of shell commands). In a python script, check for the current working directory `import os` and `print(os.getcwd())` - or using the ``pathlib`` library: `from pathlib import Path` and `print(Path.cwd())`.

In [17]:
pwd

'c:\\Users\\jschoer\\Desktop\\DSA103development\\ChemicalDataScience103\\lectures\\session10\\materials'

Robust fix: build the path from the scripts own location:

In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().resolve()    # folder of this script; 
# Path.cwd().resolve().parent would go one level higher...
data_file = BASE_DIR / "data" / "results.csv"  # "/" joins path parts

df = pd.read_csv(data_file)
df

Note that in a python script this would look differently:

```python
from pathlib import Path
import pandas as pd

BASE_DIR = Path(__file__).resolve().parent     # folder of this script
data_file = BASE_DIR / "data" / "results.csv"  # "/" joins path parts
```

### 6) Docstrings vs. single-line comments (vs. multi-line comments)

A **docstring** documents what a function, class, or module does (and is accessible at runtime). It is the first statement in a function or a class. They use triple quotes to mark their beginning and the end. 

A **comment** explains why a specific line or block of code does something (and is ignored by Python). Use the hashtag sign for comments, even if you extent to the next line (single-line comments). Multiline comments can be done using triple quotes (as the docstring), but should be avoided for clarity. Technically, it is not a comment, but a string that is created and ignored. **Instead use multiple single quote lines**.

You can toggle line comments for blocking lines from execution, see VSCode Cheatsheet! (May depend on language settings and can be changed based on your preference)

In [ ]:
def calculate_average(values):
    """Return the arithmetic mean of a list of numbers."""   # docstring
    # Guard against division by zero                          # comment
    if not values:
        return None
    return sum(values) / len(values)

**Important note**
Docstrings can be accessed e.g. by typing the function name with a subsequent question mark (or in Jupyter by hovering over it):

In [21]:
calculate_average?

Signature: calculate_average(values)
Docstring: <no docstring>
File:      ~\appdata\local\temp\ipykernel_51600\3940564545.py
Type:      function

Good practices:

- Docstring = what/how to use it, comment = why. Don’t repeat the code in words: 

```python
x = x + 1   # add 1 to x          <- useless
x = x + 1   # skip header row     <- useful (explains why)
```

- Write a docstring for every public function, class, and module, starting with a short one-line summary in imperative mood (“Return...”, “Load...”, not “Returns...” or “This function loads...”).
- One-line docstrings are fine for simple functions: """Return True if n is even."""
- Pick one docstring style and stay consistent (Google style as above, NumPy style, or reST).
- Keep comments up to date. An outdated comment is worse than none.
- Prefer clear names over comments: retention_time needs no comment, rt does.